In [1]:
from junno.j_utils import log
log

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Import

In [1]:
from importlib import reload

In [2]:
from junno.j_utils.json_template import JSONClass, JSONAttr
from junno.ovega.vega_graph import VegaGraph

## JSONClass test

In [3]:
class A(JSONClass):
    __template__ = '{"constante": 1, "groupa": {"suba": "{{a}}", "subconst": 0}, "b": "{{b}}"}'
    a = JSONAttr.String()
    b = JSONAttr.Float(list=True)
    
class C(JSONClass):
    __template__ = '{"pi": 3.14159, "a": "{{a}}", "c": "{{c}}"}'
    a = JSONAttr.List((A, str))
    c = JSONAttr(A, optional=True, default=A())

In [4]:
a = A()

In [5]:
a.a = 54
a

A
  |  a: '54'
  |  b: []

In [6]:
a['groupa']['suba'] = 7

In [7]:
a.b = [1, 5.0, '8']
a

A
  |  a: '7'
  |  b: [
  |   |  1.0,
  |   |  5.0,
  |   |  8.0
  |   | ]

In [8]:
c = C()
print(c.to_json())

{
 "pi": 3.14159,
 "a": [],
 "c": {
  "constante": 1,
  "groupa": {
   "suba": "",
   "subconst": 0
  },
  "b": []
 }
}


In [9]:
c.a.append(a)
del c.c
print(c.to_json())

{
 "pi": 3.14159,
 "a": [
  {
   "constante": 1,
   "groupa": {
    "suba": "7",
    "subconst": 0
   },
   "b": [
    1.0,
    5.0,
    8.0
   ]
  }
 ]
}


In [10]:
class B(JSONClass):
    __template__ = '{"pi": 3.1415, "{{options}}": "...", "child": {"e": 2.1, "a": "{{a}}", "{{pFloat}}": "...", "{{pBool}}": "..."}}'
    a = JSONAttr.String(optional=True,)
    options = JSONAttr.GenericProperties()
    pFloat = JSONAttr.GenericProperties('pi', float)
    pBool = JSONAttr.GenericProperties(types=bool)

In [11]:
b = B()
b.options['opt'] = 1
b.pFloat['pi'] = 2

print(b.to_json())

TypeError: 'NoneType' object is not iterable

In [13]:
del b.pFloat['pi']
b.pBool['pi'] = True
print(b.to_json())

{
 "pi": 3.1415,
 "child": {
  "e": 2.1,
  "pi": true
 },
 "opt": 1
}


In [14]:
b['child']['pi'] = False
print('b =', b.to_json())
print('b.pFloat =', b.pFloat)
print('b.pBool =', b.pBool)

b = {
 "pi": 3.1415,
 "child": {
  "e": 2.1,
  "pi": false
 },
 "opt": 1
}
b.pFloat = {}
b.pBool = {'pi': False}


In [15]:
del b['child']['pi']
b['child']['pi'] = 3.1415
print('b =', b.to_json())
print('b.pFloat =', b.pFloat)
print('b.pBool =', b.pBool)

b = {
 "pi": 3.1415,
 "child": {
  "e": 2.1,
  "pi": 3.1415
 },
 "opt": 1
}
b.pFloat = {'pi': 3.1415}
b.pBool = {}


In [16]:
b['child']['flag'] = True
print('b =', b.to_json())
b

b = {
 "pi": 3.1415,
 "child": {
  "e": 2.1,
  "flag": true,
  "pi": 3.1415
 },
 "opt": 1
}


B
  |  a: None
  |  options: {'opt': 1}
  |  pFloat: {'pi': 3.1415}
  |  pBool: {'flag': True}

## ovega test

In [17]:
spec = """{
  "$schema": "https://vega.github.io/schema/vega/v5.json",
  "padding": 5,
  "width": 500,
  "height": 300,

  "data": [
    {
      "name": "table",
      "values": [
        {"category": "A", "amount": 28},
        {"category": "B", "amount": 55},
        {"category": "C", "amount": 43},
        {"category": "D", "amount": 91},
        {"category": "E", "amount": 81},
        {"category": "F", "amount": 53},
        {"category": "G", "amount": 19},
        {"category": "H", "amount": 87}
      ]
    }
  ],

  "signals": [
    {
      "name": "tooltip",
      "value": {},
      "on": [
        {"events": "rect:mouseover", "update": "datum"},
        {"events": "rect:mouseout",  "update": "{}"}
      ]
    }
  ],

  "scales": [
    {
      "name": "xscale",
      "type": "band",
      "domain": {"data": "table", "field": "category"},
      "range": "width",
      "padding": 0.05,
      "round": true
    },
    {
      "name": "yscale",
      "domain": {"data": "table", "field": "amount"},
      "nice": true,
      "range": "height"
    }
  ],

  "axes": [
    { "orient": "bottom", "scale": "xscale" },
    { "orient": "left", "scale": "yscale" }
  ],

  "marks": [
    {
      "type": "rect",
      "from": {"data":"table"},
      "encode": {
        "enter": {
          "x": {"scale": "xscale", "field": "category"},
          "width": {"scale": "xscale", "band": 1},
          "y": {"scale": "yscale", "field": "amount"},
          "y2": {"scale": "yscale", "value": 0}
        },
        "update": {
          "fill": {"value": "steelblue"}
        },
        "hover": {
          "fill": {"value": "red"}
        }
      }
    },
    {
      "type": "text",
      "encode": {
        "enter": {
          "align": {"value": "center"},
          "baseline": {"value": "bottom"},
          "fill": {"value": "#333"}
        },
        "update": {
          "x": {"scale": "xscale", "signal": "tooltip.category", "band": 0.5},
          "y": {"scale": "yscale", "signal": "tooltip.amount", "offset": -2},
          "text": {"signal": "tooltip.amount"},
          "fillOpacity": [
            {"test": "datum === tooltip", "value": 0},
            {"value": 1}
          ]
        }
      }
    }
  ]
}"""

In [18]:
graph = VegaGraph()
graph

<IPython.core.display.Javascript object>

VegaGraph()

In [19]:
graph._spec_str = spec

In [20]:
graph._height = 300